# OpenAQ Source Contract and Warehouse Schema

## Purpose

This notebook formalizes the source contract established through the OpenAQ
API exploration.

The exploration notebook determined which source entities, locations,
parameters, sensors and historical periods are suitable for the warehouse.
This notebook converts those findings into explicit schemas, grains,
relationships and data requirements before transformation logic is developed.

It defines the contract between:

**OpenAQ source → Raw → Bronze → Silver → dimensional warehouse**

No production transformation logic is implemented here.

## Environment Setup

Import the libraries required by this notebook and load the project
configuration.

This notebook is designed to run independently and does not depend on variables
or imports created in the API exploration notebook.

In [4]:
from pathlib import Path
import sys

import pandas as pd

# Allow imports from the project root when the notebook runs from /notebooks
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Raw

Raw is the immutable source-of-truth landing layer. OpenAQ responses are
preserved in their source-aligned structure with ingestion metadata added for
lineage and replay.

Raw does not enforce the analytical schema or reject batches because source
fields or datatypes change. Unexpected source changes are preserved so they
can be detected and handled downstream.

Each landed batch records ingestion metadata including a batch identifier,
ingestion timestamp, source entity and, where applicable, the requested
extraction window.


In [3]:
# Source datasets that must be reproducible from the Raw layer
raw_source_contract = pd.DataFrame(
    [
        {
            "dataset": "countries",
            "source": "/countries",
            "landing_mode": "snapshot",
            "partition_key": "ingestion_date",
        },
        {
            "dataset": "locations",
            "source": "/locations",
            "landing_mode": "snapshot",
            "partition_key": "ingestion_date",
        },
        {
            "dataset": "sensors",
            "source": "/sensors/{sensor_id}",
            "landing_mode": "snapshot",
            "partition_key": "ingestion_date",
        },
        {
            "dataset": "parameters",
            "source": "/parameters/{parameter_id}",
            "landing_mode": "snapshot",
            "partition_key": "ingestion_date",
        },
        {
            "dataset": "measurements",
            "source": "/sensors/{sensor_id}/measurements",
            "landing_mode": "incremental",
            "partition_key": "observation_date",
        },
    ]
)

raw_source_contract

,dataset,source,landing_mode,partition_key
0,countries,/countries,snapshot,ingestion_date
1,locations,/locations,snapshot,ingestion_date
2,sensors,/sensors/{sensor_id},snapshot,ingestion_date
3,parameters,/parameters/{parameter_id},snapshot,ingestion_date
4,measurements,/sensors/{sensor_id}/measurements,incremental,observation_date


## Source Field Contract

The source field contract records the OpenAQ attributes required to construct
the warehouse while preserving known source characteristics discovered during
API exploration.

`required` indicates whether the field is structurally required by our
pipeline contract. It does not imply that every nested source value must be
non-null.

Raw preserves the source payload without enforcing these datatypes. The
expected types documented here become relevant when Raw data is structured
and validated in Bronze.

In [5]:
# Expected source structure based on the API exploration
source_field_contract = pd.DataFrame(
    [
        # Countries
        ("countries", "id", "integer", False, "Source country identifier"),
        ("countries", "code", "string", True, "Country code"),
        ("countries", "name", "string", False, "Country name"),
        ("countries", "datetimeFirst", "object", True, "First reported activity"),
        ("countries", "datetimeLast", "object", True, "Latest reported activity"),
        ("countries", "parameters", "array", True, "Available parameters"),

        # Locations
        ("locations", "id", "integer", False, "Source location identifier"),
        ("locations", "name", "string", False, "Monitoring location name"),
        ("locations", "locality", "string", True, "Locality reported by source"),
        ("locations", "timezone", "string", True, "Location timezone"),
        ("locations", "country", "object", False, "Nested country reference"),
        ("locations", "owner", "object", True, "Nested owner reference"),
        ("locations", "provider", "object", True, "Nested provider reference"),
        ("locations", "isMobile", "boolean", True, "Mobile monitoring indicator"),
        ("locations", "isMonitor", "boolean", True, "Monitor indicator"),
        ("locations", "instruments", "array", True, "Monitoring instruments"),
        ("locations", "sensors", "array", False, "Location-to-sensor relationship"),
        ("locations", "coordinates", "object", True, "Location coordinates"),
        ("locations", "licenses", "array", True, "Source licensing metadata"),
        ("locations", "bounds", "array", True, "Geographic bounds"),
        ("locations", "datetimeFirst", "object", True, "First reported activity"),
        ("locations", "datetimeLast", "object", True, "Latest reported activity"),

        # Sensors
        ("sensors", "id", "integer", False, "Source sensor identifier"),
        ("sensors", "name", "string", False, "Sensor name"),
        ("sensors", "parameter", "object", False, "Measured parameter"),
        ("sensors", "datetimeFirst", "object", True, "First sensor observation"),
        ("sensors", "datetimeLast", "object", True, "Latest sensor observation"),
        ("sensors", "coverage", "object", True, "Source-reported coverage metrics"),

        # Parameters
        ("parameters", "id", "integer", False, "Source parameter identifier"),
        ("parameters", "name", "string", False, "Parameter name"),
        ("parameters", "units", "string", False, "Measurement unit"),
        ("parameters", "displayName", "string", True, "Display name"),
        ("parameters", "description", "string", True, "Parameter description"),

        # Measurements
        ("measurements", "value", "number", False, "Observed measurement value"),
        ("measurements", "flagInfo", "object", True, "Source quality/flag information"),
        ("measurements", "parameter", "object", False, "Parameter metadata"),
        ("measurements", "period", "object", False, "Measurement time interval"),
        ("measurements", "coordinates", "object", True, "Observation coordinates"),
        ("measurements", "summary", "object", True, "Source summary metadata"),
        ("measurements", "coverage", "object", True, "Source coverage metadata"),
    ],
    columns=[
        "dataset",
        "field",
        "expected_type",
        "nullable",
        "purpose",
    ],
)

source_field_contract

,dataset,field,expected_type,nullable,purpose
0,countries,id,integer,False,Source country identifier
1,countries,code,string,True,Country code
2,countries,name,string,False,Country name
3,countries,datetimeFirst,object,True,First reported activity
4,countries,datetimeLast,object,True,Latest reported activity
5,countries,parameters,array,True,Available parameters
6,locations,id,integer,False,Source location identifier
7,locations,name,string,False,Monitoring location name
8,locations,locality,string,True,Locality reported by source
9,locations,timezone,string,True,Location timezone


### Contract Summary

The summary below provides a compact view of the number of expected and
nullable fields for each source dataset.

In [6]:
contract_summary = (
    source_field_contract
    .groupby("dataset")
    .agg(
        fields=("field", "count"),
        nullable_fields=("nullable", "sum"),
    )
    .reset_index()
)

contract_summary

,dataset,fields,nullable_fields
0,countries,6,4
1,locations,16,12
2,measurements,7,4
3,parameters,5,2
4,sensors,6,3


## Source Relationship Contract

OpenAQ exposes several warehouse relationships through nested source objects
rather than independent relational tables.

These relationships must be preserved when Raw data is structured so that
downstream dimensions and facts can be connected without reconstructing
relationships from measurement values or names.

Source identifiers remain the stable lineage keys through Raw, Bronze and
Silver. Warehouse surrogate keys can be introduced later when dimensional
models are built.

In [7]:
source_relationship_contract = pd.DataFrame(
    [
        (
            "country",
            "location",
            "locations.country.id",
            "1:N",
            "Links monitoring locations to countries"
        ),
        (
            "provider",
            "location",
            "locations.provider.id",
            "1:N",
            "Provider relationship embedded in location metadata"
        ),
        (
            "owner",
            "location",
            "locations.owner.id",
            "1:N",
            "Owner relationship embedded in location metadata"
        ),
        (
            "location",
            "sensor",
            "locations.sensors[].id",
            "1:N",
            "Authoritative location-to-sensor relationship"
        ),
        (
            "parameter",
            "sensor",
            "sensors.parameter.id",
            "1:N",
            "Identifies the parameter measured by a sensor"
        ),
        (
            "sensor",
            "measurement",
            "measurement ingestion context",
            "1:N",
            "Sensor ID comes from the measurement endpoint context",
        ),
    ],
    columns=[
        "parent_entity",
        "child_entity",
        "source_reference",
        "cardinality",
        "purpose",
    ]
)

source_relationship_contract

,parent_entity,child_entity,source_reference,cardinality,purpose
0,country,location,locations.country.id,1:N,Links monitoring locations to countries
1,provider,location,locations.provider.id,1:N,Provider relationship embedded in location met...
2,owner,location,locations.owner.id,1:N,Owner relationship embedded in location metadata
3,location,sensor,locations.sensors[].id,1:N,Authoritative location-to-sensor relationship
4,parameter,sensor,sensors.parameter.id,1:N,Identifies the parameter measured by a sensor
5,sensor,measurement,measurement ingestion context,1:N,Sensor ID comes from the measurement endpoint ...


## Warehouse Grain

The grain defines exactly what one row represents in each warehouse model.

Grain is established before transformation logic because aggregation,
deduplication and key design depend on it. Source identifiers remain available
for lineage even when surrogate warehouse keys are introduced later.

The base measurement fact preserves the finest useful observation grain.
Hourly and daily facts are derived from this base rather than replacing it.

In [8]:
# Define the intended row-level meaning of each warehouse model
warehouse_grain = pd.DataFrame(
    [
        (
            "dim_country",
            "one row per OpenAQ country",
        ),
        (
            "dim_provider",
            "one row per OpenAQ provider",
        ),
        (
            "dim_owner",
            "one row per OpenAQ owner",
        ),
        (
            "dim_location",
            "one row per OpenAQ monitoring location",
        ),
        (
            "dim_sensor",
            "one row per OpenAQ sensor",
        ),
        (
            "dim_parameter",
            "one row per OpenAQ parameter definition",
        ),
        (
            "fct_measurements",
            "one row per sensor observation and measurement period",
        ),
        (
            "fct_air_quality_hourly",
            "one row per location, parameter, unit and hour",
        ),
        (
            "fct_air_quality_daily",
            "one row per location, parameter, unit and day",
        ),
        (
            "fct_sensor_quality",
            "one row per sensor and quality evaluation period",
        ),
    ],
    columns=["model", "grain"],
)

warehouse_grain

,model,grain
0,dim_country,one row per OpenAQ country
1,dim_provider,one row per OpenAQ provider
2,dim_owner,one row per OpenAQ owner
3,dim_location,one row per OpenAQ monitoring location
4,dim_sensor,one row per OpenAQ sensor
5,dim_parameter,one row per OpenAQ parameter definition
6,fct_measurements,one row per sensor observation and measurement...
7,fct_air_quality_hourly,"one row per location, parameter, unit and hour"
8,fct_air_quality_daily,"one row per location, parameter, unit and day"
9,fct_sensor_quality,one row per sensor and quality evaluation period


## Source-to-Warehouse Mapping

Each warehouse model must be traceable to the source dataset from which its
attributes and relationships originate.

Some dimensions are constructed from nested objects rather than independent
OpenAQ endpoints. In particular, provider and owner are derived from location
metadata.

Derived fact tables are built from validated measurement data rather than
directly from the OpenAQ API.

In [9]:
# Map warehouse models to their authoritative source or downstream derivation
warehouse_source_mapping = pd.DataFrame(
    [
        (
            "dim_country",
            "countries",
            "countries.id",
            "Direct source entity"
        ),
        (
            "dim_provider",
            "locations",
            "locations.provider.id"
            "Distinct provider entities extracted from locations"
        ),
        (
            "dim_owner",
            "locations",
            "locations.owner.id",
            "Distinct owner entities extracted from locations"
        ),
        (
            "dim_location",
            "locations",
            "locations.id",
            "Direct source entity"
        ),
        (
            "dim_sensor",
            "locations + sensors",
            "sensor.id",
            "Location relationship plus sensor metadata"
        ),
        (
            "dim_parameter",
            "parameters",
            "parameters.id",
            "Direct parameter definition"
        ),
        (
            "fct_measurements",
            "measurements",
            "sensor context + measurement period",
            "Base observational fact"
        ),
        (
            "fct_air_quality_hourly",
            "fct_measurements",
            "location + parameter + unit + hour",
            "Derived hourly aggregation"
        ),
        (
            "fct_air_quality_daily",
            "fct_measurements",
            "location + parameter + unit + day",
            "Derived daily aggregation"
        ),
        (
            "fct_sensor_quality",
            "fct_measurements",
            "sensor + evaluation period",
            "Derived from observed measurement history"
        ),
    ],
    columns=[
        "warehouse_model",
        "source",
        "business_key_or_grain",
        "derivation",
    ],
)

warehouse_source_mapping

,warehouse_model,source,business_key_or_grain,derivation
0,dim_country,countries,countries.id,Direct source entity
1,dim_provider,locations,locations.provider.idDistinct provider entitie...,None
2,dim_owner,locations,locations.owner.id,Distinct owner entities extracted from locations
3,dim_location,locations,locations.id,Direct source entity
4,dim_sensor,locations + sensors,sensor.id,Location relationship plus sensor metadata
5,dim_parameter,parameters,parameters.id,Direct parameter definition
6,fct_measurements,measurements,sensor context + measurement period,Base observational fact
7,fct_air_quality_hourly,fct_measurements,location + parameter + unit + hour,Derived hourly aggregation
8,fct_air_quality_daily,fct_measurements,location + parameter + unit + day,Derived daily aggregation
9,fct_sensor_quality,fct_measurements,sensor + evaluation period,Derived from observed measurement history


## Raw Landing Contract

The Raw layer is the immutable landing zone between OpenAQ and downstream
processing.

Each successful API extraction is persisted before transformation so that
Bronze and later layers can be rebuilt without requesting the source again.

Raw landing follows four principles:

1. Preserve the complete source response wherever practical.
2. Add ingestion metadata without modifying source values.
3. Never overwrite previously landed batches.
4. Separate snapshot metadata entities from incremental measurement data.

In [10]:
# Define deterministic Raw storage paths for each source dataset
raw_landing_contract = pd.DataFrame(
    [
        (
            "countries",
            "snapshot",
            "raw/countries/ingestion_date=YYYY-MM-DD/",
        ),
        (
            "locations",
            "snapshot",
            "raw/locations/ingestion_date=YYYY-MM-DD/",
        ),
        (
            "sensors",
            "snapshot",
            "raw/sensors/ingestion_date=YYYY-MM-DD/",
        ),
        (
            "parameters",
            "snapshot",
            "raw/parameters/ingestion_date=YYYY-MM-DD/",
        ),
        (
            "measurements",
            "incremental",
            (
                "raw/measurements/"
                "sensor_id={sensor_id}/"
                "year=YYYY/month=MM/day=DD/"
            ),
        ),
    ],
    columns=[
        "dataset",
        "landing_mode",
        "path_pattern",
    ],
)

raw_landing_contract

,dataset,landing_mode,path_pattern
0,countries,snapshot,raw/countries/ingestion_date=YYYY-MM-DD/
1,locations,snapshot,raw/locations/ingestion_date=YYYY-MM-DD/
2,sensors,snapshot,raw/sensors/ingestion_date=YYYY-MM-DD/
3,parameters,snapshot,raw/parameters/ingestion_date=YYYY-MM-DD/
4,measurements,incremental,raw/measurements/sensor_id={sensor_id}/year=YY...


## Batch Lineage Contract

Every Raw object must be traceable to the ingestion run that produced it.

A logical ingestion run receives a `batch_id`. Multiple API pages or files may
share the same batch identifier while retaining their individual landing
timestamps.

Measurement batches additionally preserve the requested extraction window so
that incremental loads can be reconciled and replayed.

In [11]:
raw_lineage_contract = pd.DataFrame(
    [
        ("batch_id", "string", False, "Logical ingestion run identifier"),
        ("ingested_at", "UTC timestamp", False, "Time payload landed in Raw"),
        ("source", "string", False, "Upstream source system"),
        ("entity", "string", False, "OpenAQ source entity"),
        ("sensor_id", "integer", True, "Measurement endpoint sensor context"),
        ("window_start", "UTC timestamp", True, "Incremental extraction start"),
        ("window_end", "UTC timestamp", True, "Incremental extraction end"),
    ],
    columns=[
        "field",
        "expected_type",
        "nullable",
        "purpose",
    ],
)

raw_lineage_contract

,field,expected_type,nullable,purpose
0,batch_id,string,False,Logical ingestion run identifier
1,ingested_at,UTC timestamp,False,Time payload landed in Raw
2,source,string,False,Upstream source system
3,entity,string,False,OpenAQ source entity
4,sensor_id,integer,True,Measurement endpoint sensor context
5,window_start,UTC timestamp,True,Incremental extraction start
6,window_end,UTC timestamp,True,Incremental extraction end
